In [ ]:
# | default_exp report.articles

# Report Articles
> Sync articles to the database.

In [ ]:
# | export
from sqlmodel import Session, select
from seo_rat.article import get_article_by_path, insert_article, Article

FETCH = "::fetch::"


In [ ]:
# | export

def sync_articles_to_db(session: Session, website_id: int,
                        url_file_mapping: dict[str, str]) -> None:
    "Sync URL→file mappings into DB, preserving existing SEO metadata on re-sync."
    for url, file_path in url_file_mapping.items():
        if not file_path: continue
        try:
            if file_path == FETCH:
                existing = session.exec(select(Article).where(Article.url == url)).first()
            else:
                existing = get_article_by_path(session, file_path)
            if existing:
                if existing.url != url or existing.file_path != file_path:
                    existing.url = url
                    existing.file_path = file_path
                    session.add(existing)
            else:
                insert_article(session, Article(website_id=website_id, file_path=file_path, url=url))
        except Exception:
            pass
    session.commit()
